# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

# Video Game Character Designer Assistant

<b>This is an AI chatbot that helps you come up with video game character designs and also generates their visuals.</b>

<i>P.S. The prompts need some polishing, but technically everything works fine.</i>

In [1]:
import os
import gradio as gr
import dotenv
from openai import OpenAI
import base64
from io import BytesIO
import json
from PIL import Image

In [2]:
dotenv.load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables.")
else:
    print("OPENAI_API_KEY is set in the environment variables.")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY is not set in the environment variables.")
else:
    print("GROQ_API_KEY is set in the environment variables.")

openai = OpenAI()
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
GPT_MODEL = "gpt-4o-mini"


OPENAI_API_KEY is set in the environment variables.
GROQ_API_KEY is set in the environment variables.


In [3]:
system_message = """You are a professional video game character designer.
Your task is to help the user create a character's name, description and appearance based on the style.
Do not reply more than one sentence, be short, concise and to the point.
You create 2D illustrations."""

In [4]:
def generate_visuals(prompt):
  response = openai.images.generate(
    model="dall-e-3",
    prompt=prompt,
    n=1,
    size="1024x1024",
    response_format="b64_json",
  )
  image_base64 = response.data[0].b64_json
  image_data = base64.b64decode(image_base64)
  return Image.open(BytesIO(image_data))

In [5]:
def character_visual_generator(name, description, style):
  character_visual_system_prompt = """You are an expert prompt engineer.
  Your task is to generate a prompt that will be pasted in AI Image Generation tool to generate 2D visuals for a character in a video game.
  Your prompts are effective if the generated prompt will produce a high quality 2D image of that video game character."""
  user_prompt= f"Generate a prompt for a video game character with the following name, description and style:\nName: {name}\nDescription: {description}\nStyle: {style}"
  
  response = openai.chat.completions.create(
    model=GPT_MODEL,
    messages = [
      {"role": "system", "content": character_visual_system_prompt}, 
      {"role": "user", "content": user_prompt}
    ]
  )

  visual = generate_visuals(response.choices[0].message.content)

  return visual


In [6]:
character_visual_generator_tool = {
  "name": "character_visual_generator",
  "description": "A tool to generate visuals for a video game character based on its name, description and art style.",
  "parameters": {
    "type": "object",
    "properties": {
      "name": {
        "type": "string",
        "description": "The name of the video game character."
      },
      "description": {
        "type": "string",
        "description": "A detailed description of the video game character."
      },
      "style": {
        "type": "string",
        "description": "The art style for the character (e.g., pixel art, realistic, cartoon)."
      }
    },
    "required": ["name", "description", "style"]
  }
}

In [7]:
def speak(text):
  response = openai.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="onyx",
    input=text
  )
  return response.content

In [8]:
def transcribe(audio_file):
    if audio_file is None:
        return ""
    with open(audio_file, "rb") as f:
        transcription = openai.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe", 
            file=f
        )
    return transcription.text

In [9]:
def handle_tool_call(message):
  responses = []
  visuals = []
  for tool_call in message.tool_calls:
    if tool_call.function.name == "character_visual_generator":
      arguments = json.loads(tool_call.function.arguments)
      name = arguments.get('name')
      description = arguments.get('description')
      style = arguments.get('style')
      visual = character_visual_generator(name, description, style)
      visuals.append(visual)
      responses.append({
        "role": "tool",
        "content": "Visual has been created for the character.",
        "tool_call_id": tool_call.id
      })
  return responses, visuals

In [14]:
def chat(history):
  parse_history = [{"role": item["role"], "content": item["content"]} for item in history]
  messages = [{"role": "system", "content": system_message}] + parse_history
  visuals = []
  response = openai.chat.completions.create(
    model=GPT_MODEL,
    messages=messages,
    tools=[{"type": "function", "function": character_visual_generator_tool}]
  )

  while response.choices[0].finish_reason == "tool_calls":
    message = response.choices[0].message
    responses, visuals = handle_tool_call(message)
    messages.append(message)
    messages.extend(responses)
    response = openai.chat.completions.create(
      model=GPT_MODEL,
      messages=messages,
      tools=[{"type": "function", "function": character_visual_generator_tool}]
    )

  history += [{"role": "assistant", "content": response.choices[0].message.content}]
  voice = speak(response.choices[0].message.content)

  return history, visuals[0] if len(visuals) > 0 else None, voice

In [15]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

with gr.Blocks() as ui:
  with gr.Row():
    chatbot = gr.Chatbot(label="Chatbot", type="messages", height=500)
    visual = gr.Image(label="Visual", height=500)
  with gr.Row():
    voice = gr.Audio(label="Chatbot is speaking...", autoplay=True)
  with gr.Row():
    user_input = gr.Textbox(label="Your message", placeholder="Type your message here...")
    voice_input = gr.Audio(sources=['microphone'], type='filepath', label="Click to talk")
 
  voice_input.stop_recording(transcribe, inputs=voice_input, outputs=user_input)
  user_input.submit(put_message_in_chatbot, inputs=[user_input, chatbot], outputs=[user_input, chatbot]).then(
     chat, inputs=chatbot, outputs=[chatbot, visual, voice])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7913
* To create a public link, set `share=True` in `launch()`.
